# Figure 2D — which guide modules act on which gene modules

For every pair of a guide module and a gene module, tests whether the effect
sizes in that block are shifted relative to the effect sizes of the matrix as a
whole. Each block is tested in both directions with a Wilcoxon rank-sum test
against a random sample of the full coefficient matrix, and the p-values are
FDR corrected.

Produces `outputs/GuideModuleGeneModuleEffectSignificance.csv`.

## Setup

In [1]:
library(reshape2)

E3LIGASE <- "/home/eraslab1/Projects/E3Ligase/analysisSingle"

GUIDE_MODULES     <- file.path(E3LIGASE, "TextFiles/ME_GuideModules_leiden_6_Modules.csv")
GENE_MODULES      <- file.path(E3LIGASE, "TextFiles/ME_GeneModules_leiden_11_Modules.csv")
BETA_COEFFICIENTS <- file.path(E3LIGASE, "EffectSizes/MixedEffectLMOutputs/ME_SignificantBetaCoefs.csv")

dir.create("outputs", showWarnings = FALSE)

NULL_SAMPLE_SIZE <- 100000   # effect sizes drawn as the background for each test
FDR_THRESHOLD    <- 0.05
RANDOM_SEED      <- 1        # see the note below

## Load the modules and the effect sizes

Coefficient row names are guide identifiers; only the gene name part is kept so
they line up with the guide module table.

In [2]:
guideModules <- as.data.frame(read.csv(
    GUIDE_MODULES, stringsAsFactors = FALSE, strip.white = TRUE, header = TRUE, row.names = 1
))
geneModules <- as.data.frame(read.csv(
    GENE_MODULES, stringsAsFactors = FALSE, strip.white = TRUE, header = TRUE, row.names = 1
))
geneModules$GeneName <- rownames(geneModules)

coefsSgn <- as.data.frame(read.csv(BETA_COEFFICIENTS, header = TRUE, row.names = 1))
rownames(coefsSgn) <- sapply(rownames(coefsSgn), function(x) strsplit(x, "_")[[1]][2])

dim(coefsSgn)

[1]  329 1041

## Test every guide module against every gene module

:::{note}
The background sample is random. The original notebook drew it without setting
a seed, so its p-values could not be reproduced from one run to the next; a
seed is set here to make the result deterministic. The effect sizes and the
direction of every test are unaffected, but the p-values will not match the
original run exactly.
:::

In [3]:
set.seed(RANDOM_SEED)

# the background population is the same for every block, so build it once
allBetas <- melt(coefsSgn)$value

results <- list()
for (guideGroup in sort(unique(guideModules$NewGuideGroup))) {
    for (geneGroup in sort(unique(geneModules$GeneGroup))) {

        blockBetas <- coefsSgn[
            guideModules[guideModules$NewGuideGroup == guideGroup, "GuideName"],
            geneModules[geneModules$GeneGroup == geneGroup, "GeneName"]
        ]
        blockBetas <- melt(blockBetas)$value

        sampledBetas <- sample(allBetas, NULL_SAMPLE_SIZE)
        meanDif <- mean(blockBetas) - mean(sampledBetas)

        for (direction in c("greater", "less")) {
            test <- wilcox.test(blockBetas, sampledBetas, alternative = direction)
            results[[length(results) + 1]] <- data.frame(
                guideGroup    = guideGroup,
                geneGroup     = geneGroup,
                lessOrGreater = direction,
                P_value       = test$p.value,
                MeanDiff      = meanDif,
                stringsAsFactors = FALSE
            )
        }
    }
}

allTestRes <- do.call(rbind, results)
nrow(allTestRes)

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; using all as measure variables

No id variables; usi

[1] 132

## Correct for multiple testing and save

In [4]:
allTestRes$FDR <- p.adjust(allTestRes$P_value)
allTestRes <- allTestRes[allTestRes$FDR < FDR_THRESHOLD, ]
write.csv(allTestRes, "outputs/GuideModuleGeneModuleEffectSignificance.csv", row.names = FALSE)
allTestRes

,guideGroup,geneGroup,lessOrGreater,P_value,MeanDiff,FDR
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,M1,G0,greater,1.350107e-93,0.038504820,1.593127e-91
3,M1,G1,greater,3.074547e-56,0.028072612,3.382002e-54
6,M1,G10,less,2.441266e-18,-0.033768170,2.319203e-16
7,M1,G2,greater,4.835948e-152,0.047408822,6.044935e-150
9,M1,G3,greater,3.225481e-33,0.023443723,3.322245e-31
12,M1,G4,less,2.816648e-147,-0.046342402,3.492644e-145
14,M1,G5,less,1.433832e-54,-0.033557328,1.548538e-52
15,M1,G6,greater,1.215761e-56,0.025001296,1.349495e-54
18,M1,G7,less,0.000000e+00,-0.173689156,0.000000e+00
